# Evaluation Walkthrough

This notebook shows how the retrieval evaluation works in this project. It starts with the metric definitions, then loads the manual judgments from `data/eval_queries.example.json`, and finally runs the same evaluation code used by `src.eval`.

The live evaluation cells require Elasticsearch to be running on `localhost:9200` and the FTS index to be populated.

## What is being measured?

For ranked retrieval, the evaluator reports: 

- **Precision@k**: fraction of the top-k results that are relevant.
- **Recall@k**: fraction of all relevant results that appear in the top-k.
- **MRR**: reciprocal rank of the first relevant result.
- **nDCG@k**: ranking quality that rewards higher-ranked relevant results more.

If RAG evaluation is enabled, the notebook also scores the cited OCIDs using source coverage, source precision, and source recall.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.rag import rag_answer
from src.eval import (
    _format_per_query_details,
    _format_summary,
    _load_evaluation_queries,
    dcg_at_k,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
    reciprocal_rank,
    source_coverage,
    source_precision_at_k,
    source_recall,
    evaluate_dataset,
    evaluate_query,
)
from src.es_client import get_es_client

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Tempor

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Re

In [2]:
# A tiny toy example that shows the metric math without needing Elasticsearch.
ranked_ocids = ['a1', 'b2', 'c3', 'd4', 'e5']
relevant = {'b2', 'd4'}
gains = {'b2': 2, 'd4': 1}

print('Ranked OCIDs:', ranked_ocids)
print('Relevant OCIDs:', sorted(relevant))
print('Precision@5:', round(precision_at_k(ranked_ocids, relevant, 5), 3))
print('Recall@5:', round(recall_at_k(ranked_ocids, relevant, 5), 3))
print('MRR:', round(reciprocal_rank(ranked_ocids, relevant), 3))
print('DCG@5:', round(dcg_at_k(ranked_ocids, gains, 5), 3))
print('nDCG@5:', round(ndcg_at_k(ranked_ocids, gains, 5), 3))
print('Source coverage:', round(source_coverage(['b2', 'x9'], relevant), 3))
print('Source precision:', round(source_precision_at_k(['b2', 'x9'], relevant, 2), 3))
print('Source recall:', round(source_recall(['b2', 'x9'], relevant), 3))

Ranked OCIDs: ['a1', 'b2', 'c3', 'd4', 'e5']
Relevant OCIDs: ['b2', 'd4']
Precision@5: 0.4
Recall@5: 1.0
MRR: 0.5
DCG@5: 2.323
nDCG@5: 0.64
Source coverage: 0.5
Source precision: 0.5
Source recall: 0.5


## Load the evaluation set

The example file contains a small set of natural-language queries paired with the OCIDs that were judged relevant by hand. Those labels are the ground truth used by the evaluator.

In [3]:
queries_path = ROOT / 'data' / 'eval_queries.example.json'
evaluation_queries = _load_evaluation_queries(queries_path)

print(f'Loaded {len(evaluation_queries)} evaluation queries from {queries_path.name}\n')
for item in evaluation_queries:
    judged = ', '.join(f'{judgment.ocid}:{judgment.relevance}' for judgment in item.judgments)
    print(f'- {item.query_id}: {item.query}')
    print(f'  Judgments: {judged}')

Loaded 7 evaluation queries from eval_queries.example.json

- nhs_care_home_placements: NHS care home placements direct award
  Judgments: ocds-h6vhtk-04a3e9:1, ocds-h6vhtk-04f73c:1, ocds-h6vhtk-05ff51:1
- school_cleaning_services: school cleaning services through a DPS
  Judgments: ocds-h6vhtk-02f9a2:1, ocds-h6vhtk-05e241:1, ocds-h6vhtk-0691bc:1
- cable_run_management: cable run management systems for transport infrastructure
  Judgments: ocds-h6vhtk-051b02:1, ocds-h6vhtk-06a1b0:1, ocds-h6vhtk-051b02:1
- remote_monitoring_cameras: low power remote image collection cameras for monitoring
  Judgments: ocds-h6vhtk-038759:1, ocds-h6vhtk-038b08:1, ocds-h6vhtk-03559e:1
- utility_dynamic_market_framework: utilities dynamic market for goods works and services
  Judgments: ocds-h6vhtk-04e7a8:1, ocds-h6vhtk-0552e4:1, ocds-h6vhtk-050b32:1
- buyer_specific_tfl: Transport for London cable management contract
  Judgments: ocds-h6vhtk-051b02:1, ocds-h6vhtk-03cfe0:1, ocds-h6vhtk-05ee2a:1
- buyer_spec

## Run a live retrieval check

This cell connects to Elasticsearch and evaluates a single query with one search mode. It is a useful sanity check before running the full dataset.

In [4]:
es = get_es_client()
if not es.ping():
    raise RuntimeError('Elasticsearch is not reachable on localhost:9200. Start Docker and ingest the data first.')

single_result = evaluate_query(es, evaluation_queries[0], search_type='hybrid', k=5)
print(json.dumps(single_result, indent=2, ensure_ascii=False))

INFO:elastic_transport.transport:HEAD http://localhost:9200/ [status:200 duration:0.004s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B-Instruct-2507/cdbee75f17c01a7cc42f958dc650907174af0554/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B-Instruct-2507/cdbee75f17c01a7cc42f958dc650907174af0554/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-4B-Instruct-2507/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-4B-Instruct-2507/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-4B-Instruct-2507 "HTTP/1.1 200 OK"
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.081s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.005s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.004s]
INFO:src.es_client:No hits with strict intent filters; attempting incremental relaxation.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.123s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.305s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]


{
  "query_id": "nhs_care_home_placements",
  "query": "NHS care home placements direct award",
  "search_type": "hybrid",
  "k": 5,
  "precision_at_k": 0.0,
  "recall_at_k": 0.0,
  "mrr": 0.0,
  "ndcg_at_k": 0.0,
  "returned": 5,
  "top_ocids": [
    "ocds-h6vhtk-04a4ac",
    "ocds-h6vhtk-04a3fb",
    "ocds-h6vhtk-04a463",
    "ocds-h6vhtk-04a3e3",
    "ocds-h6vhtk-04a323"
  ],
  "parsed_intent": {
    "raw": "NHS care home placements direct award procurement",
    "filters": {
      "procurement_method": "direct award",
      "is_framework": false
    },
    "locations": [],
    "services": [
      "care home placements"
    ],
    "clean_text": "NHS care home placements direct award procurement",
    "fallback": "relaxed_procurement_method"
  }
}


## Full evaluation

This runs the evaluation loop. By default it scores `text`, `semantic`, and `hybrid` retrieval at `k=5`.

If you want to include RAG source-citation scoring, set `include_rag=True` in the next cell.

In [5]:
report = evaluate_dataset(
    evaluation_queries,
    search_types=('text', 'semantic', 'hybrid'),
    k=5,
    include_rag=True,
)

print(_format_summary(report))

[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.069s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
INFO:src.es_client:No hits with strict intent filters; attempting incremental relaxation.
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.090s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingfa

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.003s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
INFO:src.es_client:No hits with strict intent filters; attempting incremental relaxation.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.007s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.037s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
INFO:src.es_client:Relaxations failed; retrying without intent parsing.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.073s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.030s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
INFO:elastic_transport.transport:GET http://localhost:9200/fts_contracts/_mapping [status:200 duration:0.001s]


Evaluation Summary @ k=5


TEXT Search
----------------------------------------
  Queries evaluated: 7
  Precision@k:      0.257
  Recall@k:         0.452
  MRR:              0.643
  nDCG@k:           0.465
  Zero-hit queries: 0
  Intent fallbacks: relaxed_procurement_method:3, relaxed_contract_status:1

SEMANTIC Search
----------------------------------------
  Queries evaluated: 7
  Precision@k:      0.200
  Recall@k:         0.357
  MRR:              0.714
  nDCG@k:           0.440
  Zero-hit queries: 0
  Intent fallbacks: relaxed_procurement_method:3, relaxed_contract_status:1

HYBRID Search
----------------------------------------
  Queries evaluated: 7
  Precision@k:      0.314
  Recall@k:         0.548
  MRR:              0.571
  nDCG@k:           0.516
  Zero-hit queries: 0
  Intent fallbacks: relaxed_procurement_method:3, relaxed_contract_status:1

RAG Search
----------------------------------------
  Queries evaluated: 7
  Source Coverage:  0.171 (relevant sources / total cit

## Per-query breakdown

This mirrors the CLI output produced with `--detailed`, but keeps it inside the notebook so you can inspect query-by-query behavior without switching tools.

In [6]:
print(_format_per_query_details(report))

Per-Query Details


Query: buyer_specific_london_construction — Contracts awarded in London for construction
------------------------------------------------------------------------------------------------------------------------
  text       | p@k=0.20 r@k=0.33 mrr=0.25 ndcg=0.20 | top_3=['ocds-h6vhtk-03fa9f', 'ocds-h6vhtk-03fa9e', 'ocds-h6vhtk-03190e']
  semantic   | p@k=0.40 r@k=0.67 mrr=1.00 ndcg=0.77 | top_3=['ocds-h6vhtk-037dc3', 'ocds-h6vhtk-0614e5', 'ocds-h6vhtk-042251']
  hybrid     | p@k=0.40 r@k=0.67 mrr=0.50 ndcg=0.50 | top_3=['ocds-h6vhtk-03fa9f', 'ocds-h6vhtk-042cc4', 'ocds-h6vhtk-065f0e']
  rag        | cov=0.00 prec=0.00 recall=0.00 | cited=5/3 | vars=4 cands=80

Query: buyer_specific_tfl — Transport for London cable management contract
------------------------------------------------------------------------------------------------------------------------
  text       | p@k=0.40 r@k=0.67 mrr=1.00 ndcg=0.77 | top_3=['ocds-h6vhtk-05ee2a', 'ocds-h6vhtk-03cfe0', 'ocds-h6vht

## RAG pipeline internals

Instead of just the generated answer, let's examine what the RAG pipeline actually did: query expansion, filters applied, candidates evaluated, and which documents were ranked highest after deduplication.

In [7]:
# Run RAG on a single query to show the internals
test_query = evaluation_queries[1]
print(f"Evaluating: {test_query.query}\n")

rag_result = rag_answer(query=test_query.query, es=es, top_k=5)

# Extract workflow trace
workflow = rag_result.get("workflow", {})
print("=" * 80)
print("RAG PIPELINE WORKFLOW")
print("=" * 80)

print(f"\n1. Original Query:\n   {workflow.get('query', test_query.query)}")

print(f"\n2. Query Expansion:")
print(f"   Enabled: {workflow.get('expanded', False)}")
if workflow.get('query_variations'):
    print(f"   Variations generated ({len(workflow['query_variations'])} total):")
    for i, var in enumerate(workflow['query_variations'], start=1):
        print(f"     {i}. {var}")

print(f"\n3. Search Execution:")
print(f"   Searches run: {workflow.get('searches_run', 0)}")
print(f"   Candidates per query: {workflow.get('candidates_per_query', 20)}")
print(f"   Total candidates evaluated: {workflow.get('total_candidates_evaluated', 0)}")

print(f"\n4. Deduplication & Ranking:")
ranked = workflow.get('ranked_results', [])
print(f"   Documents kept after dedup: {len(ranked)}")
if ranked:
    print(f"   Top ranked documents:")
    for item in ranked:
        print(f"     [{item['rank']}] {item['title']}")
        print(f"         Buyer: {item['buyer_name']} | OCID: {item['ocid']}")

print(f"\n5. Parsed Intent:")
parsed = rag_result.get("parsed_intent", {})
if parsed:
    print(json.dumps(parsed, indent=2))
else:
    print("   (no intent filters applied)")

print("\n" + "=" * 80)
print("Generated Answer (if LLM available):")
print("=" * 80)
print(rag_result.get("answer", "(no LLM answer generated)"))

[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating: school cleaning services through a DPS



[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:src.rag:Generated 4 query variations for 'school cleaning services through a DPS'
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take prece

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.052s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
INFO:src.es_client:No hits with strict intent filters; attempting incremental relaxation.


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.059s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.005s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.001s]


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.075s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.006s]
INFO:elastic_transport.transport:POST http://localhost:9200/fts_contracts/_search [status:200 duration:0.002s]
INFO:elastic_transport.transport:GET http://localhost:9200/fts_contracts/_mapping [status:200 duration:0.002s]


RAG PIPELINE WORKFLOW

1. Original Query:
   school cleaning services through a DPS

2. Query Expansion:
   Enabled: True
   Variations generated (4 total):
     1. school cleaning services through a DPS
     2. cleaning services via framework agreement in a school setting
     3. school facility maintenance through a DPS framework procurement
     4. cleaning service contracts under a framework agreement for educational institutions

3. Search Execution:
   Searches run: 4
   Candidates per query: 20
   Total candidates evaluated: 80

4. Deduplication & Ranking:
   Documents kept after dedup: 5
   Top ranked documents:
     [1] North Baddesley Infant School - Heating Upgrade
         Buyer: Hampshire County Council | OCID: ocds-h6vhtk-0691bc
     [2] Digital Configurator
         Buyer: Department for Education | OCID: ocds-h6vhtk-0693d6
     [3] Internal Fit Out and Maintenance DPS
         Buyer: PRESTON PRIMARY ACADEMY TRUST | OCID: ocds-h6vhtk-05e241
     [4] Frozen Food
         